<a href="https://colab.research.google.com/github/HugoCrainich/Statistical-Machine-Learning/blob/main/Lab%2012/Lab_12.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [14]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
from statsmodels.tools.eval_measures import rmse
import matplotlib.pyplot as plt

# Step 1: Ingestion from external source
url = 'https://raw.githubusercontent.com/HugoCrainich/Statistical-Machine-Learning/refs/heads/main/data/Zillow_ZHVI_2026_Micro.csv'
df = pd.read_csv(url)

df.head()

,Home_Value,Square_Footage,Property_Age,Distance_to_Transit,School_District_Rating
0,329705.74,1941.0,5.5,6.45,Excellent
1,183343.63,1364.3,35.2,2.15,Average
2,354551.73,2386.9,52.4,0.75,Good
3,325773.17,2192.1,50.2,5.25,Excellent
4,359743.12,3069.8,66.5,12.69,Excellent


In [15]:
# Step 2: Defining the formula
# Utilizing the R-style patsy formula interface allows for elegant, readable model specification
formula ='Home_Value ~ Square_Footage + Property_Age + Distance_to_Transit + School_District_Rating'


In [16]:
# Step 3: Fitting the model and printing the summary
model = smf.ols(formula=formula, data=df)
results = model.fit()
print(results.summary())


                            OLS Regression Results                            
Dep. Variable:             Home_Value   R-squared:                       0.766
Model:                            OLS   Adj. R-squared:                  0.765
Method:                 Least Squares   F-statistic:                     542.5
Date:                Mon, 16 Mar 2026   Prob (F-statistic):          2.81e-309
Time:                        20:02:23   Log-Likelihood:                -12072.
No. Observations:                1000   AIC:                         2.416e+04
Df Residuals:                     993   BIC:                         2.419e+04
Df Model:                           6                                         
Covariance Type:            nonrobust                                         
                                          coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------------------------------
In

In [17]:
# Step 4: Generating predictions
# We extract the predicted values vector to transition from explanation to prediction
y_pred = results.predict(df)

In [18]:
# Step 5: Calculate RMSE between the actuals and the predictions
model_rmse = rmse(df['Home_Value'], y_pred)
print(f"\nThe Predictive RMSE is: ${model_rmse:,.2f}")


The Predictive RMSE is: $42,316.69


In [19]:
"""
Hedonic Pricing OLS Model — Interactive Residual Forensics Dashboard
=====================================================================
Prereqs: pip install statsmodels plotly pandas numpy
"""

import numpy as np
import pandas as pd
import statsmodels.api as sm
import plotly.express as px
import plotly.graph_objects as go

# ── 1. SYNTHETIC HEDONIC DATASET ─────────────────────────────────────────────
# Replace this block with your own DataFrame if you have real data.
# Hedonic variables: sqft, bedrooms, bathrooms, age, garage (binary), price.

np.random.seed(42)
n = 300

df = pd.DataFrame({
    "sqft":      np.random.randint(600, 4000, n),
    "bedrooms":  np.random.randint(1, 6, n),
    "bathrooms": np.random.randint(1, 4, n),
    "age":       np.random.randint(0, 60, n),
    "garage":    np.random.randint(0, 2, n),
})

# True price relationship (log-linear hedonic structure) + noise
df["price"] = (
    50_000
    + 120  * df["sqft"]
    + 8_000 * df["bedrooms"]
    + 12_000 * df["bathrooms"]
    - 500  * df["age"]
    + 15_000 * df["garage"]
    + np.random.normal(0, 25_000, n)   # homoscedastic baseline noise
    + np.where(df["sqft"] > 3000,      # intentional structural break:
               df["sqft"] * 80, 0)     # large homes are systematically under-priced
)

# ── 2. OLS MODEL ─────────────────────────────────────────────────────────────
# sm.add_constant() prepends a column of 1s so statsmodels estimates an
# intercept term (β₀). Without it, the regression is forced through the origin.

X = sm.add_constant(df[["sqft", "bedrooms", "bathrooms", "age", "garage"]])
y = df["price"]

model  = sm.OLS(y, X)          # define the OLS estimator
result = model.fit()           # fit via least-squares; returns RegressionResultsWrapper

print(result.summary())        # optional: inspect coefficients, R², F-stat

# ── 3. EXTRACT RESIDUALS & FITTED VALUES ─────────────────────────────────────
# result.fittedvalues  → pd.Series of ŷ (in-sample predicted prices)
# result.resid         → pd.Series of ε̂ = y - ŷ  (raw residuals)
# Both are aligned to the original index of y.

fitted    = result.fittedvalues        # ŷ: what the model predicts
residuals = result.resid               # ε̂: signed prediction error

# Root Mean Squared Error (your lab metric, reproduced here for reference)
rmse = np.sqrt(np.mean(residuals ** 2))
print(f"\nRMSE: ${rmse:,.0f}")

# ── 4. OUTLIER DETECTION ─────────────────────────────────────────────────────
# Standardise residuals: z = ε̂ / σ(ε̂)
# Any observation where |z| > 2 lies outside ~95 % of a normal distribution.
# These are flagged as potential outliers or high-leverage structural breaks.

resid_std     = residuals.std()                       # σ(ε̂) — one number
z_scores      = (residuals / resid_std).abs()         # |z| for every obs.
outlier_mask  = z_scores > 2                          # boolean Series

# Build a plotting DataFrame so Plotly has named columns to work with
plot_df = pd.DataFrame({
    "fitted":    fitted,
    "residual":  residuals,
    "z_score":   z_scores,
    "outlier":   outlier_mask.map({True: "Outlier (|z|>2)", False: "Normal"}),
    "obs_index": residuals.index,    # handy hover label
})

# ── 5. PLOTLY SCATTER — FITTED vs. RESIDUAL ───────────────────────────────────
# px.scatter builds an interactive figure; color= maps the outlier category
# to a discrete palette, which we immediately override with color_discrete_map
# to enforce crimson for outliers and a muted steel-blue for normals.

fig = px.scatter(
    plot_df,
    x            = "fitted",
    y            = "residual",
    color        = "outlier",                       # drives legend + dot colour
    color_discrete_map = {
        "Normal":           "steelblue",
        "Outlier (|z|>2)":  "crimson",              # stark crimson for outliers
    },
    hover_data   = {"obs_index": True,
                    "z_score":   ":.2f",
                    "fitted":    ":$,.0f",
                    "residual":  ":$,.0f"},
    opacity      = 0.75,
    title        = "Residual Forensics Dashboard — Fitted Values vs. Residuals",
    labels       = {
        "fitted":   "Fitted (Predicted) Price  ŷ  ($)",
        "residual": "Residual Error  ε̂ = y − ŷ  ($)",
        "outlier":  "Observation Class",
    },
    template     = "plotly_white",
)

# ── 6. ZERO-LINE ─────────────────────────────────────────────────────────────
# A horizontal line at ε̂ = 0 is the theoretical ideal: if the model is
# correctly specified, residuals should scatter randomly around this line
# with no pattern. add_hline() is the non-deprecated replacement for
# add_shape(type='line') horizontal helpers.

fig.add_hline(
    y           = 0,
    line_dash   = "dash",
    line_color  = "black",
    line_width  = 1.5,
    annotation_text      = "ε̂ = 0  (ideal)",
    annotation_position  = "bottom right",
    annotation_font_size = 11,
)

# ── 7. ANNOTATION — RMSE WATERMARK ───────────────────────────────────────────
fig.add_annotation(
    xref      = "paper", yref = "paper",
    x=0.01, y=0.97,
    text      = f"RMSE: ${rmse:,.0f}",
    showarrow = False,
    font      = dict(size=12, color="dimgray"),
    align     = "left",
)

# ── 8. LAYOUT POLISH ─────────────────────────────────────────────────────────
fig.update_layout(
    legend_title_text = "Observation Class",
    font              = dict(family="Arial", size=12),
    title_font_size   = 16,
    hovermode         = "closest",
    margin            = dict(l=60, r=30, t=70, b=60),
)

fig.update_traces(marker_size=7, marker_line_width=0.4,
                  marker_line_color="white")

fig.show()   # opens in your default browser (Jupyter: renders inline)

                            OLS Regression Results                            
Dep. Variable:                  price   R-squared:                       0.889
Model:                            OLS   Adj. R-squared:                  0.887
Method:                 Least Squares   F-statistic:                     468.9
Date:                Mon, 16 Mar 2026   Prob (F-statistic):          9.28e-138
Time:                        20:02:24   Log-Likelihood:                -3808.1
No. Observations:                 300   AIC:                             7628.
Df Residuals:                     294   BIC:                             7650.
Df Model:                           5                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const      -1.236e+05   1.93e+04     -6.387      0.0